# Telugu BPE Tokenizer Training

Trains a ByteLevel BPE tokenizer on a sampled subset of the Telugu corpus (train+val splits).

**Tokenizer specs:**
- Normalizer: NFC only (no StripAccents, no Lowercase)
- Special tokens: `<pad>`, `<unk>`, `<bos>`, `<eos>` (4 tokens for decoder-only LM)
- Vocab size: computed dynamically from corpus size via tiered heuristic
- Sampling: ~2.5GB from ~14.9GB train+val (deterministic, seed 42)
- Evaluation: round-trip encode/decode on held-out test set, regression checks

In [ ]:
import json
import logging
import random
from datetime import datetime
from pathlib import Path
from typing import Optional

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, processors, trainers

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

# Language config
LANG = "Telugu"
LANG_SHORT = "telugu"
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

# Sampling params
SAMPLE_CAP_GB = 2.5
SAMPLE_CAP_BYTES = int(SAMPLE_CAP_GB * 1024**3)
SAMPLE_SEED = 42

print(f"Training {LANG} BPE Tokenizer (with ~{SAMPLE_CAP_GB}GB sampling)")

In [ ]:
# ============================================================================
# Embedded utilities (self-contained)
# ============================================================================

def estimate_corpus_tokens(total_bytes: int, bytes_per_token: float = 4.0) -> int:
    """Rough token estimate from corpus size (bytes/4 heuristic)."""
    return int(total_bytes / bytes_per_token)


def compute_vocab_size(total_tokens_estimate: int) -> int:
    """Tiered vocab size heuristic: <50M->8K, 50M-200M->16K, 200M-1B->32K, >=1B->50K"""
    if total_tokens_estimate < 50_000_000:
        return 8_000
    elif total_tokens_estimate < 200_000_000:
        return 16_000
    elif total_tokens_estimate < 1_000_000_000:
        return 32_000
    else:
        return 50_000


def sample_lines_to_file(
    source_files: list[Path],
    target_bytes: int,
    output_path: Path,
    seed: int = 42,
) -> dict:
    """Deterministic single-pass Bernoulli line sampling."""
    total_bytes = sum(f.stat().st_size for f in source_files)
    p = min(1.0, target_bytes / total_bytes) if total_bytes else 0.0

    rng = random.Random(seed)
    written_bytes = written_lines = 0
    per_file: dict[str, int] = {}

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as out:
        for f in source_files:
            n = 0
            with open(f, "r", encoding="utf-8") as fh:
                for line in fh:
                    if rng.random() < p:
                        if not line.endswith("\n"):
                            line += "\n"
                        out.write(line)
                        written_bytes += len(line.encode("utf-8"))
                        written_lines += 1
                        n += 1
            per_file[str(f)] = n

    return {
        "sampling_probability": p,
        "source_total_bytes": total_bytes,
        "target_bytes": target_bytes,
        "actual_bytes_written": written_bytes,
        "actual_lines_written": written_lines,
        "per_file_lines_written": per_file,
        "seed": seed,
        "source_files_order": [str(f) for f in source_files],
    }

print("✓ Utility functions defined")

In [ ]:
# ============================================================================
# Tokenizer construction
# ============================================================================

def create_bpe_tokenizer() -> Tokenizer:
    """Create a ByteLevel BPE tokenizer with correct normalization."""
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
    return tokenizer


def build_trainer(vocab_size: int) -> trainers.BpeTrainer:
    """Build a BPE trainer with specified vocab size and special tokens."""
    return trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )

print("✓ Tokenizer functions defined")

In [ ]:
# ============================================================================
# Corpus discovery and preparation
# ============================================================================

def gather_training_files(split_dirs: list[Path]) -> list[Path]:
    """Find all *.txt files in given split directories, sorted."""
    files = []
    for split_dir in split_dirs:
        if split_dir.exists():
            files.extend(sorted(split_dir.glob("*.txt")))
    return files


def total_bytes(files: list[Path]) -> int:
    """Compute total size of files in bytes."""
    return sum(f.stat().st_size for f in files if f.exists())


# Set up paths (assumes notebook is in telugu/tokenizer/)
notebook_dir = Path.cwd()
LANG_ROOT = notebook_dir.parent  # -> .../telugu
DATA_DIR = LANG_ROOT / "data"
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"
TOKENIZER_DIR = notebook_dir  # -> .../telugu/tokenizer
SAMPLE_CACHE_DIR = TOKENIZER_DIR / ".sample_cache"

print(f"✓ Paths configured (LANG_ROOT={LANG_ROOT})")

In [ ]:
# ============================================================================
# Discover corpus files
# ============================================================================

logger.info("Discovering corpus files...")
train_files = gather_training_files([TRAIN_DIR])
val_files = gather_training_files([VAL_DIR])
test_files = gather_training_files([TEST_DIR])

logger.info(f"Train files: {[f.name for f in train_files]}")
logger.info(f"Val files: {[f.name for f in val_files]}")
logger.info(f"Test files: {[f.name for f in test_files]}")

# Compute vocab size from train+val
train_val_files = train_files + val_files
train_val_bytes = total_bytes(train_val_files)
train_val_tokens = estimate_corpus_tokens(train_val_bytes)
vocab_size = compute_vocab_size(train_val_tokens)

print(f"\n📊 Corpus stats (train+val):")
print(f"  Total bytes: {train_val_bytes / (1024**3):.1f} GB")
print(f"  Estimated tokens: {train_val_tokens:,}")
print(f"  Vocab size (heuristic): {vocab_size:,}")

## Sampling Phase

Telugu corpus is ~14.9GB. We'll sample down to ~2.5GB using deterministic Bernoulli line sampling (seed=42 for reproducibility).

In [ ]:
# ============================================================================
# Sample corpus
# ============================================================================

logger.info(f"Sampling {SAMPLE_CAP_BYTES / (1024**3):.1f}GB from train+val corpus...")

source_files = train_val_files
total_source_bytes = total_bytes(source_files)

logger.info(f"Source corpus: {total_source_bytes / (1024**3):.1f}GB")

sample_file = SAMPLE_CACHE_DIR / f"{LANG_SHORT}_sample_seed{SAMPLE_SEED}_{SAMPLE_CAP_GB:.1f}GB.txt"

sample_provenance = sample_lines_to_file(
    source_files,
    SAMPLE_CAP_BYTES,
    sample_file,
    seed=SAMPLE_SEED,
)

print(f"\n🎯 Sampling complete:")
print(f"  Sampled: {sample_provenance['actual_bytes_written'] / (1024**3):.2f}GB ({sample_provenance['actual_lines_written']:,} lines)")
print(f"  Sampling probability: {sample_provenance['sampling_probability']:.4f}")
print(f"  Cache file: {sample_file.name}")

In [ ]:
# ============================================================================
# Train tokenizer
# ============================================================================

logger.info("Creating tokenizer...")
tokenizer = create_bpe_tokenizer()

logger.info("Building trainer...")
trainer = build_trainer(vocab_size)

logger.info("Training BPE on sampled corpus (this may take several minutes)...")
tokenizer.train(
    files=[str(sample_file)],
    trainer=trainer,
)

print("✓ Training complete")

In [ ]:
# ============================================================================
# Save tokenizer and config
# ============================================================================

tokenizer_path = TOKENIZER_DIR / f"{LANG_SHORT}_tokenizer.json"
logger.info(f"Saving tokenizer to {tokenizer_path.name}...")
tokenizer.save(str(tokenizer_path))

vocab_actual = tokenizer.get_vocab_size()
logger.info(f"Vocab size actual: {vocab_actual:,}")
if vocab_actual != vocab_size:
    logger.warning(f"  Note: actual ({vocab_actual}) differs from requested ({vocab_size})")

# Save config
config = {
    "language": LANG,
    "tokenizer_type": "BPE",
    "model": "ByteLevel BPE",
    "normalizer": "NFC",
    "vocab_size_requested": vocab_size,
    "vocab_size_actual": vocab_actual,
    "special_tokens": SPECIAL_TOKENS,
    "sampling": {
        "cap_gb": SAMPLE_CAP_GB,
        "seed": SAMPLE_SEED,
        "actual_bytes_written": sample_provenance['actual_bytes_written'],
        "actual_lines_written": sample_provenance['actual_lines_written'],
    },
    "created_at": datetime.now().isoformat(),
}

config_path = TOKENIZER_DIR / "tokenizer_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
logger.info(f"Saved config to {config_path.name}")

print(f"✓ Tokenizer and config saved")

## Evaluation on Test Set

In [ ]:
# ============================================================================
# Evaluate tokenizer
# ============================================================================

def evaluate_tokenizer(tokenizer: Tokenizer, test_files: list[Path], sample_lines: int = 500) -> dict:
    """Evaluate tokenizer on held-out test set."""
    logger.info("Evaluating tokenizer on held-out test set...")

    sampled_lines = []
    rng = random.Random(42)
    total_read = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_read += 1
                if len(sampled_lines) < sample_lines:
                    sampled_lines.append(line)
                else:
                    j = rng.randint(0, total_read - 1)
                    if j < sample_lines:
                        sampled_lines[j] = line

    logger.info(f"Sampled {len(sampled_lines)} lines from {total_read} read")

    token_lengths = []
    char_counts = []
    token_counts = []
    unk_count = 0
    total_tokens = 0
    roundtrip_pass = 0
    example_triples = []

    for line in sampled_lines[:100]:
        encoded = tokenizer.encode(line)
        decoded = tokenizer.decode(encoded.ids)

        token_lengths.append(len(encoded.ids))
        char_counts.append(len(line))
        token_counts.append(len(encoded.ids))

        for token_id in encoded.ids:
            total_tokens += 1
            if token_id == UNK_ID:
                unk_count += 1

        if decoded == line:
            roundtrip_pass += 1

        if len(example_triples) < 3:
            example_triples.append({
                "original": line[:80],
                "num_tokens": len(encoded.ids),
                "roundtrip_ok": decoded == line,
            })

    avg_tokens_per_line = sum(token_lengths) / len(token_lengths) if token_lengths else 0
    avg_chars_per_token = sum(char_counts) / sum(token_counts) if token_counts else 0
    unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
    roundtrip_rate = 100.0 * roundtrip_pass / len(sampled_lines) if sampled_lines else 0

    return {
        "samples_evaluated": len(sampled_lines),
        "avg_tokens_per_line": round(avg_tokens_per_line, 2),
        "avg_chars_per_token": round(avg_chars_per_token, 2),
        "unk_rate_percent": round(unk_rate, 4),
        "roundtrip_match_percent": round(roundtrip_rate, 1),
        "example_triples": example_triples,
    }

eval_results = evaluate_tokenizer(tokenizer, test_files)

print(f"\n📈 Evaluation Results:")
print(f"  Samples evaluated: {eval_results['samples_evaluated']}")
print(f"  Avg tokens/line: {eval_results['avg_tokens_per_line']}")
print(f"  Avg chars/token: {eval_results['avg_chars_per_token']:.2f}")
print(f"  UNK rate: {eval_results['unk_rate_percent']:.4f}%")
print(f"  Roundtrip match: {eval_results['roundtrip_match_percent']:.1f}%")

print(f"\n📝 Example Encode/Decode:")
for i, triple in enumerate(eval_results['example_triples'], 1):
    print(f"  {i}. {triple['original'][:60]}...")
    print(f"     Tokens: {triple['num_tokens']}, Roundtrip OK: {triple['roundtrip_ok']}")

In [ ]:
# ============================================================================
# Regression tests (combining marks: vowel signs, virama)
# ============================================================================

test_cases = [
    ("క్ష", "Telugu conjunct (virama)"),
    ("కి", "Telugu vowel sign ి (U+0C3F)"),
    ("కీ", "Telugu vowel sign ీ (U+0C40)"),
    ("ద్య", "Telugu conjunct (d + virama + y)"),
]

print("\n🔍 Regression Tests (combining marks):")
all_pass = True
for text, description in test_cases:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)
    passed = (decoded == text)
    status = "✓" if passed else "✗"
    print(f"  {status} {description}")
    print(f"     Input: {text}, Decoded: {decoded}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n✓ All regression tests PASSED!")
else:
    print("\n✗ Some regression tests FAILED (check normalizer)")

## Summary

✓ Telugu tokenizer training complete!

**Outputs:**
- `telugu_tokenizer.json` - Trained tokenizer
- `tokenizer_config.json` - Configuration and metadata
- `.sample_cache/` - Cached sampled corpus (for re-training without re-sampling)